# Trabalho de Aprendizado de Máquinas

### Bibliotecas utilizadas

In [16]:
import numpy as np
from scipy.optimize import minimize
import pandas as pd
import matplotlib.pyplot as plt

### Rede Neural Regularizada

As dimensões dos parâmetros estão de acordo com as notas de aula.
$$
x = (m, n_x) \\

a_1 = (n_x + 1, m)
 \\
W_1 = (n_1, n_x +1)
\\
z_2 = (n_1,m)
\\
a_2 = (n_1 + 1, m)
\\
W_2 = (n_2, n_1+1)
\\
a_3 = (n_2,m)
\\
y = (n_2,m)
$$

In [ ]:
class Redeneural:
    def __init__(self, input_size=400, hidden_size=25, output_size=10, Lambda=1, num_iters=800, alpha=0.8):
        #Inicialização dos parâmetros da rede neural
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.Lambda = Lambda
        self.num_iters = num_iters
        self.alpha = alpha
        
        #Inicialização dos pesos (e bias na última col) com valores aleatórios, distribuição gaussiana centrada em 0 e desvio padrão de 1 (valores em torno de 0)
        epi1 = np.sqrt(6) / np.sqrt(self.input_size + self.hidden_size)
        self.W1 = np.random.rand(self.hidden_size, self.input_size + 1) * (2 * epi1) - epi1
        
        epi2 = np.sqrt(6) / np.sqrt(self.hidden_size + self.output_size)
        self.W2 = np.random.rand(self.output_size, self.hidden_size + 1) * (2 * epi2) - epi2
        
    def sigmoid(self, z):
        return 1 / (1 + np.exp(-z))

    def sigmoid_gradient(self, z):
        return self.sigmoid(z) * (1 - self.sigmoid(z))

    def forward(self, X):
        # 1. camada de entrada
        self.m = X.shape[0]
        ones = np.ones((1, self.m)) # vetor de bias para a camada oculta
        self.a1 = np.vstack((np.ones((1, self.m)), X.T)) #Camada de entrada + bias

        # 2. camada oculta
        self.z2 = self.W1 @ self.a1 
        a2_sem_bias = self.sigmoid(self.z2)
        self.a2 = np.vstack((ones, a2_sem_bias)) 

        # 3. camada de saída
        self.z3 = self.W2 @ self.a2
        self.a3 = self.sigmoid(self.z3) #saída do forward

        return self.a3.T

    def cost_function(self, y):
        self.y10 = np.zeros((self.output_size, self.m)) #one-hot encoding: sai de um vetor (dim m) para uma matriz binária (10,m).
        for i in range(1,self.output_size+1):
            self.y10[i-1, :] = np.where(y == i, 1, 0)
        J = 0
        for j in range(self.output_size):
            J = J + sum(-self.y10[j,:]*np.log(self.a3[j,:])-(1-self.y10[j,:])*np.log(1-self.a3[j,:]))
        cost = 1/self.m*J
        return cost + self.Lambda/(2*self.m)*(np.sum(self.W1[:,1:]**2)+np.sum(self.W2[:,1:]**2))
    
    def backward(self):
        grad1 = np.zeros(self.W1.shape)
        grad2 = np.zeros(self.W2.shape)
        for j in range(self.m):
            a1j = self.a1[:,j]
            a2j = self.a2[:,j]
            delta3 = self.a3[:,j] - self.y10[:,j]
            delta2 = self.W2[:,1:].T @ delta3 * self.sigmoid_gradient(self.z2[:,j])
            grad1 = grad1 + delta2[:,np.newaxis]@a1j[:,np.newaxis].T
            grad2 = grad2 + delta3[:,np.newaxis]@a2j[:,np.newaxis].T
                                 
        grad1 = 1/self.m*grad1
        grad2 = 1/self.m*grad2
                                    
        self.grad1_reg = grad1 + (self.Lambda/self.m)*np.hstack((np.zeros((self.W1.shape[0],1)),self.W1[:,1:]))
        self.grad2_reg = grad2 + (self.Lambda/self.m)*np.hstack((np.zeros((self.W2.shape[0],1)),self.W2[:,1:]))
        return self.grad1_reg, self.grad2_reg

    def GradientDescent(self, X, y):
        self.W = np.concatenate((self.W1.flatten(), self.W2.flatten()))
        J_history = []

        for i in range(self.num_iters):
            n_W1 = (self.input_size+1)*self.hidden_size

            self.W1 = self.W[:n_W1].reshape(self.hidden_size, self.input_size+1)
            self.W2 = self.W[n_W1:].reshape(self.output_size, self.hidden_size+1)

            self.forward(X)
            J_reg = self.cost_function(y)
            grad1_reg, grad2_reg = self.backward()

            self.W = np.concatenate((
                (self.W1 - self.alpha*grad1_reg).flatten(),
                (self.W2 - self.alpha*grad2_reg).flatten()
            ))

            J_history.append(J_reg)

        self.W1 = self.W[:n_W1].reshape(self.hidden_size, self.input_size+1)
        self.W2 = self.W[n_W1:].reshape(self.output_size, self.hidden_size+1)

        return self.W, J_history

    

    def trainAll(self, X, y): 
        #Função que treina a rede neural com todos os dados de uma vez, sem separar em treino, teste e validação
        self.GradientDescent(X, y)

        output = self.forward(X)
        predictions = np.argmax(output, axis=1) + 1

        accuracy = np.sum(predictions == y) / X.shape[0] * 100

        print(f"Acurácia: {accuracy:.2f}%")

    def trainSep(self, X_train, y_train, X_val, y_val, X_test, y_test): #falta calcular o erro em cada conjunto de dados
        # separa em treino, teste e validação
        self.GradientDescent(X_train, y_train)

        output_train = self.forward(X_train)
        predictions_train = np.argmax(output_train, axis=1) + 1
        accuracy_train = np.sum(predictions_train == y_train) / X_train.shape[0] * 100

        output_test = self.forward(X_test)
        predictions_test = np.argmax(output_test, axis=1) + 1
        accuracy_test = np.sum(predictions_test == y_test) / X_test.shape[0] * 100

        output_val = self.forward(X_val)
        predictions_val = np.argmax(output_val, axis=1) + 1
        accuracy_val = np.sum(predictions_val == y_val) / X_val.shape[0] * 100

        print(f"A Acurácia e o erro no conjunto de treino foram, respectivamente: {accuracy_train:.2f}%")
        print(f"A Acurácia e o erro no conjunto de teste foram, respectivamente: {accuracy_test:.2f}%")
        print(f"A Acurácia e o erro no conjunto de validação foram, respectivamente: {accuracy_val:.2f}%")
        

### Leitura dos dados MNIST

Casas decimais estavam representadas por vírgula, o que necessitou um ajuste na leitura deles.

In [13]:
import pandas as pd

X = pd.read_csv(
    "imageMNIST.csv",
    decimal=",",
    low_memory=False
).values.astype(float)

y = pd.read_csv(
    "labelMNIST.csv",
    decimal=",",
    low_memory=False
).values.flatten()

print(X.shape)
print(X.dtype)
print(y.shape)
print(y.dtype)

(4999, 400)
float64
(4999,)
int64


In [ ]:
fig,axis = plt.subplots(10,10,figsize=(8,8))
for i in range(10):
    for j in range(10):
        axis[i,j].imshow(X[np.random.randint(0,5001),:].reshape(20,20,order="F"),cmap="hot")
        axis[i,j].axis("off")

Divisão da base de dados

In [14]:
def split_data(X, y, test_size=0.2, val_size=0.2):
    # Embaralhar os dados
    if len(X) != len(y):
        raise ValueError("O tamanho de X e y deve ser o mesmo.")
    tam = X.shape[0]
    indices = np.arange(tam)
    np.random.shuffle(indices)
    X = X[indices]
    y = y[indices]
    X_train = X[:int((1 - (test_size + val_size)) * tam)]
    y_train = y[:int((1 - (test_size + val_size)) * tam)]
    X_val = X[int((1 - (test_size + val_size)) * tam):int((1 - test_size) * tam)]
    y_val = y[int((1 - (test_size + val_size)) * tam):int((1 - test_size) * tam)]
    X_test = X[int((1 - test_size) * tam):]
    y_test = y[int((1 - test_size) * tam):]

    return X_train, X_val, X_test, y_train, y_val, y_test

## Treinamento da rede


In [ ]:
rede = Redeneural()

Parte I: Classificação de todas as imagens.

In [ ]:
rede.trainAll(X, y)

Parte II: Classificação com divisão na base de dados

In [17]:
X_train, X_val, X_test, y_train, y_val, y_test = split_data(X, y)
